## Project Warehouse

Carte : Grille 20×20, étagères d'1 case, 20 étagères placées aléatoirement au reset.

Contrainte de génération : un BFS doit vérifier que la zone de livraison et au moins une case adjacente libre de chaque étagère sont atteignables. Sinon, on régénère la carte.

Livraison : position fixe sur un bord.
Avec 6 robots, une seule case crée un goulot, donc prévoir une zone de 2 à 3 cases.

Robots : 6 robots, positions et orientations aléatoires au spawn, sur des cases libres distinctes.
Capacité 1 colis, vitesse 1 case par pas.

Colis : 10 colis actifs en permanence, au plus 1 par étagère.
À chaque livraison, un nouveau colis apparaît sur une étagère libre tirée au hasard.
Chaque colis a un âge, remis à 0 au spawn.

Actions (6 par robot) : Action / Effet Avancer / Reculer : déplacement d'1 case selon l'orientation
Tourner G / D : rotation de 90° sur place Prendre : valide si le robot est vide, adjacent à l'étagère, orienté face à elle, et que l'étagère porte un colis
Déposer : valide si le robot est chargé et face à (ou sur) la zone de livraison Une action invalide ne fait rien et coûte une pénalité Si deux robots prennent le même colis au même pas, on tire au sort celui qui l'obtient.

In [11]:
import random
import numpy as np
from dataclasses import dataclass
from collections import deque

In [16]:
def parse(text):
    """Map string -> grid, a tuple of equal-length row strings."""
    rows = tuple(line.strip() for line in text.strip().splitlines())
    if not rows or any(len(r) != len(rows[0]) for r in rows):
        raise ValueError("map rows must be non-empty and all the same width")
    bad = set("".join(rows)) - {AISLE, SHELF, DOCK}
    if bad:
        raise ValueError(f"unknown map characters: {sorted(bad)}")
    return rows


def cells_of(grid, kind):
    """All (x, y) cells of one kind, in row-major order so indices are stable."""
    return [(x, y) for y, row in enumerate(grid) for x, c in enumerate(row) if c == kind]


def in_bounds(grid, x, y):
    return 0 <= y < len(grid) and 0 <= x < len(grid[0])


def is_walkable(grid, x, y):
    return in_bounds(grid, x, y) and grid[y][x] == AISLE


def access_cells(grid, cell):
    """Walkable neighbours from which a robot can interact with `cell`."""
    x, y = cell
    return [(x + dx, y + dy) for dx, dy in DIRS if is_walkable(grid, x + dx, y + dy)]

def bfs(grid, start, goals, blocked=frozenset()):
    """Shortest walkable path from start to the nearest goal, excluding start.

    Returns [] if start is already a goal, None if no goal is reachable. Neighbours are
    expanded in DIRS order so the result is deterministic.
    """
    goals = set(goals)
    if start in goals:
        return []
    parent = {start: None}
    frontier = deque([start])
    while frontier:
        cur = frontier.popleft()
        for dx, dy in DIRS:
            nxt = (cur[0] + dx, cur[1] + dy)
            if nxt in parent or nxt in blocked or not is_walkable(grid, *nxt):
                continue
            parent[nxt] = cur
            if nxt in goals:
                path = [nxt]
                while parent[path[-1]] != start:
                    path.append(parent[path[-1]])
                return path[::-1]
            frontier.append(nxt)
    return None

def validate_map(grid, n_robots=6):
    """Raise ValueError if the warehouse map is invalid."""

    width = len(grid[0])
    height = len(grid)

    if width != 20 or height != 20:
        raise ValueError(
            f"expected a 20x20 map, got {width}x{height}"
        )

    shelves = cells_of(grid, SHELF)
    docks = cells_of(grid, DOCK)
    aisles = cells_of(grid, AISLE)

    if len(shelves) != 20:
        raise ValueError(
            f"expected 20 shelves, got {len(shelves)}"
        )

    if not docks:
        raise ValueError("map needs at least one dock")

    if len(aisles) < n_robots:
        raise ValueError(
            "not enough free cells for the robots"
        )

    # All aisle cells must be connected.
    reachable = flood_fill(grid, aisles[0])

    if len(reachable) != len(aisles):
        raise ValueError(
            "not all aisle cells are connected"
        )

    # Every shelf needs at least one accessible aisle cell.
    for shelf in shelves:
        if not access_cells(grid, shelf):
            raise ValueError(
                f"shelf {shelf} has no accessible neighbour"
            )

    # Every dock needs at least one accessible aisle cell.
    for dock in docks:
        if not access_cells(grid, dock):
            raise ValueError(
                f"dock {dock} has no accessible neighbour"
            )

def flood_fill(grid, start):
    """Return all walkable cells reachable from start."""

    visited = {start}
    stack = [start]

    while stack:
        x, y = stack.pop()

        for dx, dy in DIRS:
            nxt = (x + dx, y + dy)

            if nxt in visited:
                continue

            if not is_walkable(grid, *nxt):
                continue

            visited.add(nxt)
            stack.append(nxt)

    return visited

def problems(grid, n_robots):
    """Reasons the map is unusable; an empty list means it is valid."""
    found = []
    aisles = cells_of(grid, AISLE)
    shelves, docks = cells_of(grid, SHELF), cells_of(grid, DOCK)
    if not shelves or not docks:
        found.append("map needs at least one shelf and one dock")
    # Robots need room to start apart and to get out of each other's way.
    if len(aisles) < 2 * n_robots:
        found.append(f"only {len(aisles)} aisle cells for {n_robots} robots")
    if aisles and len(_flood(grid, aisles[0])) != len(aisles):
        found.append("aisle cells are not all connected")
    for cell in shelves + docks:
        if not access_cells(grid, cell):
            found.append(f"{cell} has no walkable neighbour")
    return found

def random_layout(seed, width=None, height=None, n_robots=2):
    """A random warehouse-shaped map string, validated for `n_robots`.

    Shape: an aisle along the top, blocks of shelves (1-2 wide, 2 deep) separated by
    aisles, then an aisle row and a bottom row holding the docks. Blocks are at most 2x2
    so every shelf cell touches an aisle.
    """
    rng = random.Random(seed)
    for _ in range(100):
        w = width or rng.randint(7, 11)
        h = height or rng.randint(6, 9)
        rows = [[AISLE] * w for _ in range(h)]
        y = 1
        while y + 1 <= h - 3:
            x = 1
            while x <= w - 2:
                block_w = min(rng.choice((1, 2)), w - 1 - x)
                for bx in range(x, x + block_w):
                    rows[y][bx] = rows[y + 1][bx] = SHELF
                x += block_w + rng.choice((1, 1, 2))
            y += 3
        for dx in rng.sample(range(w), rng.randint(1, max(1, w // 4))):
            rows[h - 1][dx] = DOCK
        text = "\n".join("".join(r) for r in rows)
        if not problems(parse(text), n_robots):
            return text
    raise ValueError(f"no valid layout found for seed={seed}, width={width}, height={height}")

In [17]:
AISLE, SHELF, DOCK = ".", "S", "D"

# Index matches the movement actions: 0 up, 1 down, 2 left, 3 right (y grows downward).
DIRS = ((0, -1), (1, 0), (0, 1), (-1, 0))

ORIENTATION = ["UP", "RIGHT", "DOWN", "LEFT"]

DEFAULT_MAP = """
....................
....................
......SSSSS.........
....................
....................
..............S.....
..............S.....
..............S.....
..............S.....
..............S.....
.......SS...........
....................
....................
..S.................
..S.................
..S.................
....................
............SSSSS...
....................
........DDD.........
"""

# Actions

ACTIONS = ["FORWARD", "BACKWARD", "TURN_LEFT", "TURN_RIGHT", "PICKUP", "DROPOFF"]

N_ACTIONS = 6

# Cell types

CELLS = ["AISLE", "WALL", "SHELF_EMPTY", "SHELF_PACKAGE", "DOCK"]

# Rewards

REWARD_STEP = 0.0
REWARD_INVALID = -1.0
REWARD_DROPOFF = 10.0

def reward_pickup(package):
  a = package.age
  if a > 100:
    return 0.0
  else:
    return (100 - a) / 10

# Robots

@dataclass
class Robot:
    position: tuple[int, int]
    orientation: int # 0 up, 1 down, 2 left, 3 right to match DIRS
    carrying: int | None = None

# Packages

@dataclass
class Package:
    id: int
    position: tuple[int, int] | None
    age: int = 0
    carried_by: int | None = None


In [26]:
class Warehouse:
    def __init__(
        self,
        layout=DEFAULT_MAP,
        n_robots=6,
        n_packages=10,
        max_steps=500,
        seed=None,
    ):
        self.grid = parse(layout)

        self.width = len(self.grid[0])
        self.height = len(self.grid)

        self.n_robots = n_robots
        self.n_packages = n_packages
        self.max_steps = max_steps

        validate_map(self.grid, n_robots=self.n_robots)

        self.shelves = cells_of(self.grid, SHELF)
        self.docks = cells_of(self.grid, DOCK)

        self.rng = random.Random(seed)

        self.robots = []
        self.packages = {}

        self.timestep = 0
        self.delivered = 0

        self.reset(seed)

    def reset(self, seed=None):
        if seed is not None:
            self.rng.seed(seed)

        self.timestep = 0
        self.delivered = 0

        # Spawn robots
        free_cells = cells_of(self.grid, AISLE)
        robot_positions = self.rng.sample(free_cells, self.n_robots)

        self.robots = [
            Robot(
                position=position,
                orientation=self.rng.randrange(4),
                carrying=None,
            )
            for position in robot_positions
        ]

        # Spawn packages
        package_shelves = self.rng.sample(
            self.shelves,
            self.n_packages,
        )

        self.packages = {}
        for package_id, shelf in enumerate(package_shelves):
            self.packages[package_id] = Package(
                id=package_id,
                position=shelf,
                age=0,
                carried_by=None,
            )

        return [
            self.observe(robot_id)
            for robot_id in range(self.n_robots)
        ]

    def package_at(self, position):
        """Return the package at position, or None."""
        for package in self.packages.values():
            if package.position == position:
                return package
        return None

    def _spawn_package(self):
        occupied_shelves = {
            package.position
            for package in self.packages.values()
            if package.position is not None
        }

        available_shelves = [
            shelf
            for shelf in self.shelves
            if shelf not in occupied_shelves
        ]

        if not available_shelves:
            raise RuntimeError(
                "no shelf available for new package"
            )

        shelf = self.rng.choice(available_shelves)

        new_id = (
            max(self.packages.keys(), default=-1)
            + 1
        )

        self.packages[new_id] = Package(
            id=new_id,
            position=shelf,
            age=0,
            carried_by=None,
        )

    def move_allowed(self, position):
        x, y = position
        return (
            0 <= x < self.width
            and 0 <= y < self.height
            and self.grid[y][x] == AISLE
        )

    def pickup(self, robot_id):
        robot = self.robots[robot_id]

        if robot.carrying is not None:
            return REWARD_INVALID

        dx, dy = DIRS[robot.orientation]
        position = (
            robot.position[0] + dx,
            robot.position[1] + dy,
        )

        package = self.package_at(position)

        if package is None:
            return REWARD_INVALID

        robot.carrying = package.id
        package.position = None
        package.carried_by = robot_id
        return reward_pickup(package)

    def dropoff(self, robot_id):
        robot = self.robots[robot_id]

        if robot.carrying is None:
            return REWARD_INVALID

        dx, dy = DIRS[robot.orientation]
        position = (
            robot.position[0] + dx,
            robot.position[1] + dy,
        )

        if position not in self.docks:
            return REWARD_INVALID

        package = self.packages[robot.carrying]
        del self.packages[robot.carrying]

        robot.carrying = None
        self.delivered += 1

        self._spawn_package()

        return REWARD_DROPOFF

    def move(self, action, robot_id):
        robot = self.robots[robot_id]

        if action == "FORWARD":
            dx, dy = DIRS[robot.orientation]
            new_position = (
                robot.position[0] + dx,
                robot.position[1] + dy,
            )

            if not self.move_allowed(new_position):
                return REWARD_INVALID

            robot.position = new_position
            return REWARD_STEP

        if action == "BACKWARD":
            dx, dy = DIRS[robot.orientation]
            new_position = (
                robot.position[0] - dx,
                robot.position[1] - dy,
            )

            if not self.move_allowed(new_position):
                return REWARD_INVALID

            robot.position = new_position
            return REWARD_STEP

        if action == "TURN_LEFT":
            robot.orientation = (robot.orientation - 1) % 4
            return REWARD_STEP

        if action == "TURN_RIGHT":
            robot.orientation = (robot.orientation + 1) % 4
            return REWARD_STEP

        if action == "PICKUP":
            return self.pickup(robot_id)

        if action == "DROPOFF":
            return self.dropoff(robot_id)

        return REWARD_INVALID

    def observe(self, robot_id):
        robot = self.robots[robot_id]

        return {
            "position": robot.position,
            "orientation": robot.orientation,
            "carrying": robot.carrying is not None,
            "neighbors": self._observe_neighbors(robot),
            "packages": tuple(
                sorted(
                    package.position
                    for package in self.packages.values()
                    if package.position is not None
                )
            ),
            "docks": tuple(self.docks),
        }

    def _observe_neighbors(self, robot):
        # Return the cell types of the 4 neighboring directions
        neighbors = []
        x, y = robot.position
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if not (0 <= nx < self.width and 0 <= ny < self.height):
                neighbors.append("WALL")
            elif (nx, ny) in self.docks:
                neighbors.append("DOCK")
            elif (nx, ny) in self.shelves:
                if self.package_at((nx, ny)) is not None:
                    neighbors.append("SHELF_PACKAGE")
                else:
                    neighbors.append("SHELF_EMPTY")
            else:
                neighbors.append("AISLE")
        return tuple(neighbors)

In [12]:
def _observe_neighbors(self, robot):
    # Return the cell types of the 4 neighboring directions
    neighbors = []
    x, y = robot.position
    for dx, dy in DIRS:
        nx, ny = x + dx, y + dy
        if not (0 <= nx < self.width and 0 <= ny < self.height):
            neighbors.append("WALL")
        elif (nx, ny) in self.docks:
            neighbors.append("DOCK")
        elif (nx, ny) in self.shelves:
            if self.package_at((nx, ny)) is not None:
                neighbors.append("SHELF_PACKAGE")
            else:
                neighbors.append("SHELF_EMPTY")
        else:
            neighbors.append("AISLE")
    return tuple(neighbors)

# Dynamically inject the missing method into the Warehouse class
Warehouse._observe_neighbors = _observe_neighbors

In [31]:
def get_state(obs):
    """
    Maps the comprehensive observation features into a unique discrete state index.
    Features included:
    - Position: (x, y) coordinates on a 20x20 grid -> 400 states
    - Orientation: 4 directions (0 to 3) -> 4 states
    - Carrying: True/False (1 or 0) -> 2 states
    - Neighbors (Up, Right, Down, Left): Each can be AISLE, WALL, SHELF_EMPTY, SHELF_PACKAGE, or DOCK -> 5^4 = 625 states

    Total states (nb_state) = 400 * 4 * 2 * 625 = 2,000,000
    """
    x, y = obs["position"]
    pos_idx = y * 20 + x  # 0 to 399

    orientation = obs["orientation"]  # 0 to 3
    carrying = 1 if obs["carrying"] else 0  # 0 or 1

    # Map each neighbor string to a discrete value from 0 to 4
    neighbor_map = {"AISLE": 0, "WALL": 1, "SHELF_EMPTY": 2, "SHELF_PACKAGE": 3, "DOCK": 4}
    n_up = neighbor_map.get(obs["neighbors"][0], 0)
    n_right = neighbor_map.get(obs["neighbors"][1], 0)
    n_down = neighbor_map.get(obs["neighbors"][2], 0)
    n_left = neighbor_map.get(obs["neighbors"][3], 0)

    # Base 5 encoding for the 4 neighbors
    neighbors_idx = n_up * 125 + n_right * 25 + n_down * 5 + n_left  # 0 to 624

    # Combine all dimensions into a unique flat index
    state_val = (
        (pos_idx * 4 * 2 * 625) +
        (orientation * 2 * 625) +
        (carrying * 625) +
        neighbors_idx
    )
    return state_val

In [20]:
def render(self):
  rows = [
      list(row)
      for row in self.grid
  ]

  # Show shelves.
  for shelf in self.shelves:

      x, y = shelf

      if self.package_at(shelf) is not None:
          rows[y][x] = "P"
      else:
          rows[y][x] = "S"

  # Show docks.
  for dock in self.docks:

      x, y = dock
      rows[y][x] = "D"

  # Show robots.
  for i, robot in enumerate(self.robots):

      x, y = robot.position

      if robot.carrying is not None:
          symbol = chr(
              ord("A") + i
          )
      else:
          symbol = str(i)

      rows[y][x] = symbol

  print("\n".join(
      "".join(row)
      for row in rows
  ))

  print(
      f"step = {self.timestep}/{self.max_steps}"
  )

  print(
      f"delivered = {self.delivered}"
  )

  for i, robot in enumerate(self.robots):

      print(
          f"robot {i}: "
          f"position={robot.position}, "
          f"orientation={robot.orientation}, "
          f"carrying={robot.carrying}"
      )

Warehouse.render = render

In [32]:
#Choose action

def choose_action(state, Q):
    if np.random.random() < epsilon:
        return np.random.choice(N_ACTIONS)

    return np.argmax(Q[state])


def learnWithQLearning(Q, state, state2, reward, action, alpha, gamma):
    predict = Q[state, action]

    Q[state, action] += alpha * (reward + gamma * np.max(Q[state2]) - predict)

In [ ]:
alpha = 0.1
gamma = 0.9
epsilon = 1

# Updated to match the unique state representation space
nb_state = 2000000

Q = [
    np.zeros((nb_state, N_ACTIONS))
    for _ in range(6)
]

env = Warehouse(
    layout=DEFAULT_MAP,
    n_robots=6,
    n_packages=10,
    max_steps=1000,
    seed=42,
)

for episode in range(1000):

    env.reset()

    done = False
    step = 0

    while not done and step < 1000:

        for robot_id in range(env.n_robots):

            # Current state
            state = get_state(env.observe(robot_id))

            # Choose action
            action = choose_action(state, Q[robot_id])

            # Move robot
            reward = env.move(
                ACTIONS[action],
                robot_id
            )

            # New state
            state2 = get_state(env.observe(robot_id))

            # Q-learning
            learnWithQLearning(Q[robot_id], state, state2, reward, action, alpha, gamma)

        step += 1

    epsilon = max(0.01, epsilon - 0.0001)

    print(
        "Episode:", episode,
        "Deliveries:", env.delivered
    )

Episode: 0 Deliveries: 1
Episode: 1 Deliveries: 0
Episode: 2 Deliveries: 0
Episode: 3 Deliveries: 0
Episode: 4 Deliveries: 0
Episode: 5 Deliveries: 0
Episode: 6 Deliveries: 0
Episode: 7 Deliveries: 0
Episode: 8 Deliveries: 0
Episode: 9 Deliveries: 0
Episode: 10 Deliveries: 0
Episode: 11 Deliveries: 2
Episode: 12 Deliveries: 1
Episode: 13 Deliveries: 0
Episode: 14 Deliveries: 1
Episode: 15 Deliveries: 1
Episode: 16 Deliveries: 0
Episode: 17 Deliveries: 0
Episode: 18 Deliveries: 1
Episode: 19 Deliveries: 1
Episode: 20 Deliveries: 0
Episode: 21 Deliveries: 2
Episode: 22 Deliveries: 2
Episode: 23 Deliveries: 0
Episode: 24 Deliveries: 0
Episode: 25 Deliveries: 3
Episode: 26 Deliveries: 0
Episode: 27 Deliveries: 0
Episode: 28 Deliveries: 0
Episode: 29 Deliveries: 1
Episode: 30 Deliveries: 3
Episode: 31 Deliveries: 0
Episode: 32 Deliveries: 1
Episode: 33 Deliveries: 1
Episode: 34 Deliveries: 0
Episode: 35 Deliveries: 3
Episode: 36 Deliveries: 1
Episode: 37 Deliveries: 1
Episode: 38 Deliveries

In [30]:
print(Q)

[array([[ 0.93225907,  0.67317759,  1.07977985,  1.09227375,  0.11499979,
         0.18345665],
       [ 0.91509322,  1.82008155,  1.21330355,  1.19241822,  0.1225002 ,
         0.11506914],
       [ 0.84106482,  0.78954453,  0.5401291 ,  0.54130378, -0.43196507,
        -0.48729841],
       [ 1.33864255,  1.45328843,  3.33319571,  3.41156285,  4.07736436,
         2.50746492],
       [ 2.04834227,  1.94969978,  4.55648331,  4.60739812,  5.4181369 ,
         3.52760875],
       [ 1.51030087,  1.43739386,  3.07519733,  2.83392523,  4.58409048,
         1.89883328],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.20108288, -0.17103881,  0.08614707,  0.08293846, -0.90905943,
        -0.90803185],
       [-0.14292446, -0.07836177,  0.01146688,  0.01163736, -0.98856554,
        -0.98812565],
       [ 2.1169201 ,  3.62126619,  4.82353514,  4.90668615,  4.20995798,
         5.91542413]]), array([[ 1.12677534e+00,  3.58968902e-01,  2.99